### Day 1 - Validate Against Historical Fire

In [1]:
import pandas as pd
import joblib

In [2]:
df = pd.read_csv('../data/merged_fire_data.csv')

In [3]:
weather_df = pd.read_csv('../data/weather_data.csv')

In [4]:
best_rf = joblib.load('../models/wildfire_spread_model.pkl')

In [5]:
df.shape

(4071, 25)

In [6]:
weather_df = weather_df.rename(columns={"time": "datetime"})

In [7]:
weather_df.shape

(984, 5)

In [8]:
df['latitude'].between(59, 61).unique()

array([ True])

In [9]:
df['longitude'].between(-121, -119).unique()

array([ True])

In [10]:
df['week'].between(34, 39).unique()

array([ True])

In [11]:
feature_columns = ['latitude', 'longitude', 'brightness', 'frp', 'bright_t31', 'temperature_2m', 
                    'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'elevation', 'slope', 'vegetation', 
                    'wind_slope_alignment', 'vpd', 'fuel_dryness',
                    'confidence']

In [12]:
y = df['fire_spread']

In [13]:
X = df[feature_columns]

In [14]:
X.isnull().sum()

latitude                0
longitude               0
brightness              0
frp                     0
bright_t31              0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
elevation               0
slope                   0
vegetation              0
wind_slope_alignment    0
vpd                     0
fuel_dryness            0
confidence              0
dtype: int64

In [15]:
predicted_spread = best_rf.predict(X)

In [16]:
df['predicted_spread'] = predicted_spread

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [18]:
accuracy = accuracy_score(df['fire_spread'], df['predicted_spread'])

In [19]:
f"{round(accuracy * 100, 2)}%"

'98.55%'

In [20]:
precision = precision_score(df['fire_spread'], df['predicted_spread'])

In [21]:
f"{round(precision * 100, 2)}%"

'98.84%'

In [22]:
recall = recall_score(df['fire_spread'], df['predicted_spread'])

In [23]:
f"{round(recall * 100, 2)}%"

'98.63%'

#### 2. Visualize Validation Results on a Map
  * Actual spread - rows where fire_spread == 1 plotted in red
  * Predicted spread - rows where predicted_spread == 1 plotted in blue
  * Misses - rows where actual and predicted disagree plotted in yellow

In [24]:
# 1. Create a Folium map centered on your case study region - latitude 60, longitude -120

In [25]:
import folium

In [26]:
validation_map = folium.Map(location=[60, -120], zoom_start=7)

In [27]:
# 2. Loop through your filtered dataframe and add a CircleMarker for each row colored by category

In [28]:
df

,latitude,longitude,brightness,acq_date,acq_time,satellite,confidence,bright_t31,frp,daynight,...,wind_direction_10m,elevation,slope,vegetation,wind_slope_alignment,vpd,fuel_dryness,fire_spread_risk,fire_spread,predicted_spread
0,60.4007,-120.4775,312.8,2023-08-21,443,Terra,86,282.8,20.0,N,...,211,563.746343,0.701510,10,-9.337368,0.810324,0.289286,39.369722,0,0
1,59.9441,-119.5471,329.8,2023-08-21,443,Terra,100,282.9,40.4,N,...,211,580.000000,1.118034,0,-14.881471,0.810324,0.289286,126.746243,0,1
2,59.9373,-119.4737,313.1,2023-08-21,443,Terra,75,285.4,18.5,N,...,211,581.000000,0.707107,0,-9.411869,0.810324,0.289286,36.707555,1,1
3,59.9447,-119.3600,321.8,2023-08-21,443,Terra,100,283.4,27.0,N,...,211,584.000000,2.061553,0,-27.440076,0.810324,0.289286,156.191340,1,1
4,59.9283,-119.3926,335.7,2023-08-21,443,Terra,100,290.2,48.2,N,...,211,585.000000,0.707107,0,-9.411869,0.810324,0.289286,95.638061,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4066,59.5749,-120.0687,317.1,2023-09-23,2108,Aqua,77,288.9,17.2,D,...,20,486.000000,4.272002,10,22.997013,1.435912,0.563636,173.943489,0,0
4067,59.6793,-119.6101,317.9,2023-09-23,2108,Aqua,78,287.5,18.3,D,...,20,546.000000,0.707107,0,3.806493,1.435912,0.563636,30.632637,0,0
4068,59.5771,-120.0504,322.9,2023-09-23,2108,Aqua,82,288.3,23.1,D,...,20,508.000000,0.500000,10,2.691597,1.435912,0.563636,27.342000,0,0
4069,59.7215,-119.6373,351.3,2023-09-23,2108,Aqua,96,290.9,69.8,D,...,20,552.000000,0.500000,0,2.691597,1.435912,0.563636,82.617818,0,0


In [29]:
for _, row in df.iterrows():
  latitude_deg = row['latitude']
  longitude_deg = row['longitude']
  point_color = "red" if (row['fire_spread'] == 1 and row['predicted_spread'] == 1) else "blue" if (row['fire_spread'] == 0 and row['predicted_spread'] == 0) else "yellow" if (row['fire_spread'] == 1 and row['predicted_spread'] == 0) else "orange"
  folium.CircleMarker(location=[latitude_deg, longitude_deg], radius=4, color=point_color, fill=True).add_to(validation_map)

In [30]:
# 3. Save as validation_map.html in your notebooks/ folder

In [31]:
validation_map.save("validation_map.html")

### Day 3 - Build the Fire Risk Model

In [32]:
# Step 1 - Generate random coordinates 
  # Use NumPy to generate random latitude values between 59 and 61, and random longitude values between -121 and -119.
  # Generate the same number of random points as you have fire detections - so around 4,071 points This gives you a balanced dataset.

In [33]:
import numpy as np

In [34]:
south_boundary, north_boundary, west_boundary, east_boundary = 59, 61, -121, -119

In [35]:
random_number_generator = np.random.default_rng(seed=1234)

In [36]:
latitudes = random_number_generator.uniform(low=south_boundary, high=north_boundary, size=(df.shape[0],)).tolist()
longitudes = random_number_generator.uniform(low=west_boundary, high=east_boundary, size=(df.shape[0],)).tolist()

In [37]:
len(latitudes), len(longitudes)

(4071, 4071)

In [38]:
latitudes[:5]

[60.953399533396286,
 59.76039147003924,
 60.84649246752791,
 59.52338484772709,
 59.638194116828394]

In [39]:
longitudes[:5]

[-119.43617808728925,
 -119.42157454008564,
 -119.59994609675219,
 -119.79885128857978,
 -119.84142839331332]

In [40]:
# Step 2 - Attach a datetime to each point
  # Each negative sample needs a timestamp so you can attach weather data to it. Randomly sample datetimes 
  # from the same date range as your fire detections - August 21, 2023 to September 30, 2023

In [41]:
datetime_col = df['datetime']

In [42]:
random_datetime_samples = random_number_generator.choice(datetime_col, size=df.shape[0], replace=True)

In [43]:
random_datetime_samples

array(['2023-08-29 05:00:00', '2023-08-26 05:00:00',
       '2023-09-01 19:00:00', ..., '2023-08-28 06:00:00',
       '2023-08-28 11:00:00', '2023-09-01 19:00:00'],
      shape=(4071,), dtype=object)

In [44]:
# Step 3 - Attach weather data
  # For each negative sample's datetime, look up the corresponding weather conditions from your weather_df -- 
  # temperature, humidity, wind speed, wind direction

In [45]:
negative_samples = pd.DataFrame({
    'latitude': latitudes,
    'longitude': longitudes,
    'datetime': random_datetime_samples
})

In [46]:
negative_samples_weather = pd.merge(negative_samples, weather_df, how="left", on="datetime")

In [47]:
negative_samples_weather.shape

(4071, 7)

In [48]:
negative_samples_weather.columns.tolist()

['latitude',
 'longitude',
 'datetime',
 'temperature_2m',
 'relative_humidity_2m',
 'wind_speed_10m',
 'wind_direction_10m']

In [49]:
negative_samples_weather.head()

,latitude,longitude,datetime,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m
0,60.953400,-119.436178,2023-08-29 05:00:00,20.4,58,5.8,248
1,59.760391,-119.421575,2023-08-26 05:00:00,19.8,42,13.8,195
2,60.846492,-119.599946,2023-09-01 19:00:00,21.9,39,29.0,263
3,59.523385,-119.798851,2023-09-01 22:00:00,22.6,31,30.2,271
4,59.638194,-119.841428,2023-08-26 05:00:00,19.8,42,13.8,195


In [50]:
# Step 4 - Attach terrain data
  # For each negative sample's latitude and longitude, look up elevation, slope, and vegetation
  # using the same rasterio approach from Week 2

In [51]:
import rasterio

In [52]:
band1_float = np.load('../data/elevation.npy')

In [53]:
slope = np.load('../data/slope.npy')

In [54]:
vegetation_clipped = np.load('../data/vegetation.npy')

In [55]:
with rasterio.open("../data/elevation.tif") as elevation:
    elevation_rows, elevation_cols = rasterio.transform.rowcol(elevation.transform, negative_samples_weather['longitude'], negative_samples_weather['latitude'])

In [56]:
elevation_rows.min(), elevation_rows.max(), elevation_cols.min(), elevation_cols.max()

(np.int32(1), np.int32(7200), np.int32(1), np.int32(7200))

In [57]:
np.clip(elevation_rows, a_min=0, a_max=7199, out=elevation_rows)
np.clip(elevation_cols, a_min=0, a_max=7199, out=elevation_cols)

array([5630, 5682, 5040, ..., 6786, 2215, 2153],
      shape=(4071,), dtype=int32)

In [58]:
elevation_rows.min(), elevation_rows.max(), elevation_cols.min(), elevation_cols.max()

(np.int32(1), np.int32(7199), np.int32(1), np.int32(7199))

In [59]:
negative_samples_weather['elevation'] = band1_float[elevation_rows, elevation_cols]

#### Adding Slope Data

In [60]:
negative_samples_weather['slope'] = slope[elevation_rows, elevation_cols]

In [61]:
negative_samples_weather['elevation'].isnull().sum(), negative_samples_weather['elevation'].isna().sum()

(np.int64(2067), np.int64(2067))

In [62]:
negative_samples_weather['slope'].isnull().sum(), negative_samples_weather['slope'].isna().sum()

(np.int64(2068), np.int64(2068))

In [63]:
negative_samples_weather['elevation'] = negative_samples_weather['elevation'].fillna(negative_samples_weather['elevation'].mean())

In [64]:
negative_samples_weather['slope'] = negative_samples_weather['slope'].fillna(negative_samples_weather['slope'].mean())

In [65]:
from rasterio.merge import merge

In [66]:
with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N57W123_Map.tif') as vegetation_south:
  with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W120_Map.tif') as vegetation_east:
    with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W123_Map.tif') as vegetation:
      merged_array, merged_transform = merge([vegetation, vegetation_east, vegetation_south])
      vegetation_rows, vegetation_cols = rasterio.transform.rowcol(merged_transform, negative_samples_weather['longitude'], negative_samples_weather['latitude'])

In [67]:
vegetation_rows.min(), vegetation_rows.max(), vegetation_cols.min(), vegetation_cols.max()

(np.int32(24004), np.int32(47998), np.int32(24003), np.int32(47999))

In [68]:
vegetation_row_start, vegetation_column_start = rasterio.transform.rowcol(merged_transform, west_boundary, north_boundary)
vegetation_row_end, vegetation_column_end = rasterio.transform.rowcol(merged_transform, east_boundary, south_boundary)

In [69]:
vegetation_rows.min(), vegetation_rows.max(), vegetation_rows.shape

(np.int32(24004), np.int32(47998), (4071,))

In [70]:
vegetation_cols.min(), vegetation_cols.max(), vegetation_cols.shape

(np.int32(24003), np.int32(47999), (4071,))

In [71]:
adjusted_rows = vegetation_rows - vegetation_row_start
adjusted_cols = vegetation_cols - vegetation_column_start

In [72]:
adjusted_rows.min(), adjusted_rows.max(), adjusted_cols.min(), adjusted_cols.max()

(np.int32(4), np.int32(23998), np.int32(3), np.int32(23999))

In [73]:
negative_samples_weather['vegetation'] = vegetation_clipped[adjusted_rows, adjusted_cols]

In [74]:
# Step 5 - Label and combine
  # Label all negative samples as 0. Label all your fire detections as 1. 
  # Combine into a single dataframe and you have you training data for the risk model.

In [75]:
# 1. Add a fire_risk column to negative_samples_weather and set all values to 0 - these are non-fire locations

In [76]:
negative_samples_weather['fire_risk'] = 0

In [77]:
# 2. Add a fire_risk column to fire detection dataframe (merged_fire_df) and set all values to 1 - these are fire locations

In [78]:
df['fire_risk'] = 1

In [79]:
# 3. Select the same columns from both dataframes so they match

In [80]:
df_subset = df[negative_samples_weather.columns]

In [81]:
# 4. Combine them into a single dataframe using pd.concat()

In [82]:
risk_model_df = pd.concat([df_subset, negative_samples_weather])

In [83]:
# 5. Print the shape and class distribution

In [84]:
risk_model_df.shape

(8142, 11)

In [85]:
risk_model_df['fire_risk'].value_counts()

fire_risk
1    4071
0    4071
Name: count, dtype: int64

### Day 4 - Train and evaluate the risk model

In [86]:
# 1. Select your feature columns (weather and terrain only)

In [87]:
features = ['latitude', 'longitude', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',  'wind_direction_10m', 'elevation', 'slope', 'vegetation']

In [88]:
# 2. Define X and y - X is your features, y is fire_risk

In [89]:
X = risk_model_df[features]

In [90]:
y = risk_model_df['fire_risk']

In [91]:
# 3. Split into 80/20 train test split with reproducible results

In [92]:
from sklearn.model_selection import train_test_split

In [93]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

In [94]:
# 4. Train a baseline Logistic Regression

In [95]:
from sklearn.preprocessing import StandardScaler

In [96]:
standardScaler = StandardScaler()

In [97]:
X_train_scaled = standardScaler.fit_transform(X_train)

In [98]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features)

In [99]:
X_test_scaled = standardScaler.transform(X_test)

In [100]:
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features)

In [101]:
from sklearn.linear_model import LogisticRegression

In [102]:
logistic_regression = LogisticRegression()

In [103]:
logistic_regression.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [104]:
y_test_predictions = logistic_regression.predict(X_test_scaled)

In [105]:
f'{round(accuracy_score(y_test, y_test_predictions) * 100, 2)}%'

'59.18%'

In [106]:
# 5. Train a Random Forest

In [107]:
from sklearn.ensemble import RandomForestClassifier

In [108]:
rf = RandomForestClassifier()

In [109]:
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [110]:
rf_test_predictions = rf.predict(X_test)

In [111]:
rf_accuracy = accuracy_score(y_test, rf_test_predictions) * 100

In [112]:
f'{round(rf_accuracy, 2)}'

'91.59'

In [113]:
from sklearn.metrics import classification_report, confusion_matrix

In [114]:
cr = classification_report(y_test, rf_test_predictions)
print(cr)

              precision    recall  f1-score   support

           0       0.95      0.87      0.91       822
           1       0.88      0.96      0.92       807

    accuracy                           0.92      1629
   macro avg       0.92      0.92      0.92      1629
weighted avg       0.92      0.92      0.92      1629



In [115]:
cm = confusion_matrix(y_test, rf_test_predictions)
print(cm)

[[719 103]
 [ 34 773]]


In [116]:
# 6. Print feature importance

In [117]:
features = pd.DataFrame(rf.feature_importances_, index=features)[0]

In [118]:
features.sort_values(ascending=False)

latitude                0.309552
elevation               0.251808
slope                   0.144531
longitude               0.133926
vegetation              0.056237
temperature_2m          0.031036
wind_direction_10m      0.026812
relative_humidity_2m    0.023256
wind_speed_10m          0.022843
Name: 0, dtype: float64

In [119]:
risk_model_df.to_csv('../data/risk_model_data.csv', index=False)

In [120]:
joblib.dump(rf, '../models/fire_risk_random_forest_model.pkl')

['../models/fire_risk_random_forest_model.pkl']

### Day 5 - Combine both models into a simple pipeline

In [121]:
import joblib
import pandas as pd

In [122]:
fire_spread_model = joblib.load('../models/wildfire_spread_model.pkl')
fire_risk_model = joblib.load('../models/fire_risk_random_forest_model.pkl')

In [123]:
# 2. Define a function that takes location and weather inputs

In [124]:
risk_feature_columns = ['latitude', 'longitude', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',  'wind_direction_10m', 'elevation', 'slope', 'vegetation']

In [125]:
spread_feature_columns = ['latitude', 'longitude', 'brightness', 'frp', 'bright_t31', 'temperature_2m', 
                    'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'elevation', 'slope', 'vegetation', 
                    'wind_slope_alignment', 'vpd', 'fuel_dryness', 'fire_spread_risk', 
                    'confidence']

In [126]:
union = set(risk_feature_columns) | set(spread_feature_columns)

In [127]:
union

{'bright_t31',
 'brightness',
 'confidence',
 'elevation',
 'fire_spread_risk',
 'frp',
 'fuel_dryness',
 'latitude',
 'longitude',
 'relative_humidity_2m',
 'slope',
 'temperature_2m',
 'vegetation',
 'vpd',
 'wind_direction_10m',
 'wind_slope_alignment',
 'wind_speed_10m'}

In [128]:
def predict_fire_behavior(latitude, longitude, relative_humidity, slope, temperature, vegetation, vpd, wind_direction, wind_slope_alignment, wind_speed, bright_t31, brightness, confidence, elevation, frp, fuel_dryness):
    risk_feature_vector = pd.DataFrame({
        'latitude': [latitude], 
        'longitude': [longitude], 
        'temperature_2m': [temperature], 
        'relative_humidity_2m': [relative_humidity], 
        'wind_speed_10m': [wind_speed],  
        'wind_direction_10m': [wind_direction], 
        'elevation': [elevation], 
        'slope': [slope], 
        'vegetation': [vegetation]
    })
    
    spread_feature_vector = pd.DataFrame({
        'latitude': [latitude], 
        'longitude': [longitude], 
        'brightness': [brightness], 
        'frp': [frp], 
        'bright_t31': [bright_t31], 
        'temperature_2m': [temperature],
        'relative_humidity_2m': [relative_humidity], 
        'wind_speed_10m': [wind_speed], 
        'wind_direction_10m': [wind_direction], 
        'elevation': [elevation], 
        'slope': [slope], 
        'vegetation': [vegetation], 
        'wind_slope_alignment': [wind_slope_alignment], 
        'vpd': [vpd], 
        'fuel_dryness': [fuel_dryness],
        'confidence': [confidence]
    })
    fire_spread_predictions = fire_spread_model.predict_proba(spread_feature_vector)
    fire_risk_predictions = fire_risk_model.predict_proba(risk_feature_vector)
    return fire_spread_predictions[0][1], fire_risk_predictions[0][1]

In [129]:
# 3. Runs both models and returns both predictions

In [130]:
import random

In [131]:
merged_fire_df = pd.read_csv('../data/merged_fire_data.csv')

In [132]:
highest_frp_row = merged_fire_df.loc[merged_fire_df['frp'].idxmax()]
lowest_frp_row = merged_fire_df.loc[merged_fire_df['frp'].idxmin()]
random_row = merged_fire_df.sample(1, random_state=1234).iloc[0]
spread_row = merged_fire_df[merged_fire_df['fire_spread'] == 1].iloc[0]
non_spread_row = merged_fire_df[merged_fire_df['fire_spread'] == 0].iloc[0]

In [133]:
highest_frp_row

latitude                            59.8859
longitude                         -119.8053
brightness                            504.1
acq_date                         2023-09-01
acq_time                               2211
satellite                              Aqua
confidence                              100
bright_t31                            317.9
frp                                 12997.7
daynight                                  D
type                                      0
week                                     35
datetime                2023-09-01 22:00:00
temperature_2m                         22.6
relative_humidity_2m                     31
wind_speed_10m                         30.2
wind_direction_10m                      271
elevation                             521.0
slope                                   0.5
vegetation                                0
wind_slope_alignment             -14.834169
vpd                                1.892105
fuel_dryness                    

In [134]:
lowest_frp_row

latitude                            59.9401
longitude                         -119.0083
brightness                            413.8
acq_date                         2023-09-01
acq_time                               2033
satellite                              Aqua
confidence                              100
bright_t31                            302.8
frp                                     0.0
daynight                                  D
type                                      0
week                                     35
datetime                2023-09-01 21:00:00
temperature_2m                         22.9
relative_humidity_2m                     32
wind_speed_10m                         31.7
wind_direction_10m                      271
elevation                             602.0
slope                              0.707107
vegetation                                0
wind_slope_alignment              -22.02067
vpd                                1.898893
fuel_dryness                    

In [135]:
random_row

latitude                            59.1038
longitude                         -120.6272
brightness                            350.0
acq_date                         2023-09-01
acq_time                               2033
satellite                              Aqua
confidence                               96
bright_t31                            299.6
frp                                    74.8
daynight                                  D
type                                      0
week                                     35
datetime                2023-09-01 21:00:00
temperature_2m                         22.9
relative_humidity_2m                     32
wind_speed_10m                         31.7
wind_direction_10m                      271
elevation                             663.0
slope                                   0.5
vegetation                               10
wind_slope_alignment             -15.570965
vpd                                1.898893
fuel_dryness                    

In [136]:
spread_row

latitude                            59.9373
longitude                         -119.4737
brightness                            313.1
acq_date                         2023-08-21
acq_time                                443
satellite                             Terra
confidence                               75
bright_t31                            285.4
frp                                    18.5
daynight                                  N
type                                      0
week                                     34
datetime                2023-08-21 05:00:00
temperature_2m                         16.2
relative_humidity_2m                     56
wind_speed_10m                          9.7
wind_direction_10m                      211
elevation                             581.0
slope                              0.707107
vegetation                                0
wind_slope_alignment              -9.411869
vpd                                0.810324
fuel_dryness                    

In [137]:
non_spread_row

latitude                            60.4007
longitude                         -120.4775
brightness                            312.8
acq_date                         2023-08-21
acq_time                                443
satellite                             Terra
confidence                               86
bright_t31                            282.8
frp                                    20.0
daynight                                  N
type                                      0
week                                     34
datetime                2023-08-21 05:00:00
temperature_2m                         16.2
relative_humidity_2m                     56
wind_speed_10m                          9.7
wind_direction_10m                      211
elevation                        563.746343
slope                               0.70151
vegetation                               10
wind_slope_alignment              -9.337368
vpd                                0.810324
fuel_dryness                    

In [138]:
# 4. Test it on a few sample inputs

In [139]:
#     return fire_spread_predictions[0][1], fire_risk_predictions[0][1]

In [140]:
r = {"Highest FRP": highest_frp_row, "Lowest FRP": lowest_frp_row, "Random Row": random_row, "Spread Case": spread_row, "No Spread Case": non_spread_row}

In [141]:
for key in r:
    spread_pred, risk_pred = predict_fire_behavior(r[key]['latitude'], r[key]['longitude'], r[key]['relative_humidity_2m'],
                                                   r[key]['slope'], r[key]['temperature_2m'], r[key]['vegetation'], 
                                                   r[key]['vpd'], r[key]['wind_direction_10m'], r[key]['wind_slope_alignment'],
                                                   r[key]['wind_speed_10m'], r[key]['bright_t31'], r[key]['brightness'],
                                                   r[key]['confidence'], r[key]['elevation'], r[key]['frp'], r[key]['fuel_dryness'])
    print(f"{key}. Spread Prediction: {round(spread_pred * 100, 2)}%. Risk Prediction: {round(risk_pred * 100, 2)}% ")

Highest FRP. Spread Prediction: 99.33%. Risk Prediction: 91.0% 
Lowest FRP. Spread Prediction: 17.67%. Risk Prediction: 60.0% 
Random Row. Spread Prediction: 100.0%. Risk Prediction: 98.0% 
Spread Case. Spread Prediction: 58.2%. Risk Prediction: 94.0% 
No Spread Case. Spread Prediction: 26.23%. Risk Prediction: 100.0% 


### Day 6 - Final visualizations

In [142]:
# 1. Create a risk zone map - plot your case study region colored by fire risk probability from your risk model

In [143]:
increment = 0.05

In [144]:
num_latitudes = int((north_boundary - south_boundary) / increment) + 1

In [145]:
latitude_degrees = np.linspace(south_boundary, north_boundary, num=num_latitudes)

In [146]:
east_boundary, west_boundary

(-119, -121)

In [147]:
num_longitudes = int((east_boundary - west_boundary) / increment) + 1

In [148]:
longitude_degrees = np.linspace(west_boundary, east_boundary, num=num_longitudes)

In [149]:
longitude_degrees

array([-121.  , -120.95, -120.9 , -120.85, -120.8 , -120.75, -120.7 ,
       -120.65, -120.6 , -120.55, -120.5 , -120.45, -120.4 , -120.35,
       -120.3 , -120.25, -120.2 , -120.15, -120.1 , -120.05, -120.  ,
       -119.95, -119.9 , -119.85, -119.8 , -119.75, -119.7 , -119.65,
       -119.6 , -119.55, -119.5 , -119.45, -119.4 , -119.35, -119.3 ,
       -119.25, -119.2 , -119.15, -119.1 , -119.05, -119.  ])

In [150]:
lat, long = np.meshgrid(latitude_degrees, longitude_degrees)

In [151]:
grid_latitudes, grid_longitudes = lat.flatten(), long.flatten()

In [152]:
grid_latitudes

array([59.  , 59.05, 59.1 , ..., 60.9 , 60.95, 61.  ], shape=(1681,))

In [153]:
grid_longitudes

array([-121., -121., -121., ..., -119., -119., -119.], shape=(1681,))

In [154]:
grid_df = pd.DataFrame({
    "latitude": grid_latitudes,
    "longitude": grid_longitudes
})

In [155]:
mean_temperature = weather_df['temperature_2m'].mean()
mean_humidity = weather_df['relative_humidity_2m'].mean()
mean_wind_speed = weather_df['wind_speed_10m'].mean()
mean_wind_direction = weather_df['wind_direction_10m'].mean()

In [156]:
grid_df['temperature_2m'] = mean_temperature
grid_df['relative_humidity_2m'] = mean_humidity
grid_df['wind_speed_10m'] = mean_wind_speed
grid_df['wind_direction_10m'] = mean_wind_direction

In [157]:
# Attach terrain

In [158]:
import rasterio

In [159]:
from rasterio.merge import merge

In [160]:
grid_df

,latitude,longitude,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m
0,59.00,-121.0,13.730285,64.480691,10.04624,197.947154
1,59.05,-121.0,13.730285,64.480691,10.04624,197.947154
2,59.10,-121.0,13.730285,64.480691,10.04624,197.947154
3,59.15,-121.0,13.730285,64.480691,10.04624,197.947154
4,59.20,-121.0,13.730285,64.480691,10.04624,197.947154
...,...,...,...,...,...,...
1676,60.80,-119.0,13.730285,64.480691,10.04624,197.947154
1677,60.85,-119.0,13.730285,64.480691,10.04624,197.947154
1678,60.90,-119.0,13.730285,64.480691,10.04624,197.947154
1679,60.95,-119.0,13.730285,64.480691,10.04624,197.947154


In [161]:
with rasterio.open('../data/elevation.tif') as elevation:
    row, col = rasterio.transform.rowcol(elevation.transform, grid_df['longitude'], grid_df['latitude'])

In [162]:
band1_float = np.load('../data/elevation.npy')

In [163]:
row.min(), row.max(), col.min(), col.max()

(np.int32(0), np.int32(7200), np.int32(0), np.int32(7200))

In [164]:
np.clip(row, 0, 7199, out=row)
np.clip(col, 0, 7199, out=col)

array([   0,    0,    0, ..., 7199, 7199, 7199],
      shape=(1681,), dtype=int32)

In [165]:
row.min(), row.max(), col.min(), col.max()

(np.int32(0), np.int32(7199), np.int32(0), np.int32(7199))

In [166]:
elevation_values = band1_float[row, col]

In [167]:
grid_df['elevation'] = elevation_values

In [168]:
slope = np.load('../data/slope.npy')

In [169]:
slope_values = slope[row, col]

In [170]:
grid_df['slope'] = slope_values

In [171]:
# vegetation

In [172]:
with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N57W123_Map.tif') as vegetation_south:
  with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W120_Map.tif') as vegetation_east:
    with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W123_Map.tif') as vegetation:
      merged_array, merged_transform = merge([vegetation, vegetation_east, vegetation_south])

In [173]:
vegetation = np.load('../data/vegetation.npy')

In [174]:
v_rows, v_cols = rasterio.transform.rowcol(merged_transform, grid_df['longitude'], grid_df['latitude'])

In [175]:
v_rows.min(), v_rows.max(), v_cols.min(), v_cols.max()

(np.int32(24000), np.int32(48000), np.int32(24000), np.int32(48000))

In [176]:
v_row_start, v_col_start = rasterio.transform.rowcol(merged_transform, west_boundary, north_boundary)

In [177]:
v_row_end, v_col_end = rasterio.transform.rowcol(merged_transform, east_boundary, south_boundary)

In [178]:
v_row_start, v_col_start, v_row_end, v_col_end

(np.int32(24000), np.int32(24000), np.int32(48000), np.int32(48000))

In [179]:
adjusted_v_rows = v_rows - v_row_start
adjusted_v_cols = v_cols - v_col_start

In [180]:
adjusted_v_rows.min(), adjusted_v_rows.max(), adjusted_v_cols.min(), adjusted_v_cols.max()

(np.int32(0), np.int32(24000), np.int32(0), np.int32(24000))

In [181]:
np.clip(adjusted_v_rows, 0, 23999, out=adjusted_v_rows)
np.clip(adjusted_v_cols, 0, 23999, out=adjusted_v_cols)

array([    0,     0,     0, ..., 23999, 23999, 23999],
      shape=(1681,), dtype=int32)

In [182]:
adjusted_v_rows.min(), adjusted_v_rows.max(), adjusted_v_cols.min(), adjusted_v_cols.max()

(np.int32(0), np.int32(23999), np.int32(0), np.int32(23999))

In [183]:
grid_df['vegetation'] = vegetation[adjusted_v_rows, adjusted_v_cols]

In [ ]:
grid_df.isnull().sum()

latitude                  0
longitude                 0
temperature_2m            0
relative_humidity_2m      0
wind_speed_10m            0
wind_direction_10m        0
elevation               820
slope                   861
vegetation                0
dtype: int64

In [197]:
grid_df['elevation'] = grid_df['elevation'].fillna(grid_df['elevation'].mean())

In [198]:
grid_df['slope'] = grid_df['slope'].fillna(grid_df['slope'].mean())

In [190]:
risk_feature_columns

['latitude',
 'longitude',
 'temperature_2m',
 'relative_humidity_2m',
 'wind_speed_10m',
 'wind_direction_10m',
 'elevation',
 'slope',
 'vegetation']

In [201]:
risk_probabilities = fire_risk_model.predict_proba(grid_df[risk_feature_columns])[:, 1]

In [203]:
grid_df['risk_probability'] = risk_probabilities

In [ ]:
# 2. Plot the risk map

In [205]:
import folium

In [208]:
import matplotlib.colors as mcolors

In [213]:
import matplotlib.cm as colormap

In [231]:
risk_map = folium.Map(location=[60, -120], zoom_start=8)

In [232]:
normalizer = mcolors.Normalize(vmin=0, vmax=1)

In [233]:
for _, row in grid_df.iterrows():
    risk_probability = normalizer(row['risk_probability'])
    rgba_color = colormap.YlOrRd(risk_probability)
    hex_color = mcolors.to_hex(rgba_color)
    folium.CircleMarker(location=[row['latitude'], row['longitude']], radius=8, color=hex_color, fill=True).add_to(risk_map)

In [234]:
merged_fire_df_sample = merged_fire_df.sample(n=1000, replace=False, random_state=1234)

In [235]:
for _, row in merged_fire_df_sample.iterrows():
    folium.CircleMarker(location=[row['latitude'], row['longitude']], radius=8, color="blue", fill=True).add_to(risk_map)

In [236]:
# 4. Save as HTMl file

In [237]:
risk_map.save("risk_map.html")